In [ ]:

# ============================================
# CELL 1: Install Dependencies
# ============================================
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [ ]:
# ============================================
# CELL 2: Import Libraries
# ============================================
import json
from typing import List, Dict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from transformers import get_scheduler, DataCollatorWithPadding
from datasets import Dataset, DatasetDict
from scipy.stats import pearsonr
from tqdm.auto import tqdm
import math
import re
import requests
import evaluate

In [ ]:
# ============================================
# CELL 3: Data Loading Functions
# ============================================
def load_jsonl_url(url):
    """Fetches and parses a JSONL file from a URL."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = [json.loads(line) for line in response.text.strip().split('\n') if line]
        return data
    except Exception as e:
        print(f"Error loading JSONL from {url}: {e}")
        return None

def transform_sighan_data(external_data, language="zho", domain="restaurant"):
    """
    Transforms the SIGHAN 2024 data structure (a list) into the
    "Quadruplet" format your code expects.
    """
    transformed_data = []

    for entry in external_data:
        quadruplet_list = []

        # Loop through the parallel lists in the SIGHAN data
        for i in range(len(entry["Aspect"])):
            quad = {
                "Aspect": entry["Aspect"][i],
                "Category": entry["Category"][i],
                "Opinion": entry["Opinion"][i],
                "VA": entry["Intensity"][i] # Already in "V#A" format
            }
            quadruplet_list.append(quad)

        # Build the final dictionary in the target format
        new_entry = {
            "ID": entry["ID"],
            "Text": entry["Sentence"], # Map "Sentence" to "Text"
            "Quadruplet": quadruplet_list # This is the key your code expects
        }
        transformed_data.append(new_entry)

    return transformed_data
def load_json_url(url):
    """
    Fetches and parses a single, complete JSON file from a URL.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json() # Use .json() for a single JSON object/list
        return data
    except Exception as e:
        print(f"Error loading JSON from {url}: {e}")
        return None

In [ ]:
# ============================================
# CELL 4: Load Raw Data
# ============================================
subtask = "subtask_3"
task = "task3"
langs = ["eng", "zho", "jpn", "rus", "tat", "ukr"]
domains = ["restaurant", "laptop", "finance", "hotel"]

all_train = []
all_test = []  # For final predictions

print("--- Loading DimABSA 2026 Data ---")
for lang in langs:
    for domain in domains:
        # Finance domain uses task1 for training, others use alltasks
        if domain == "finance":
            specified_task = "task1"
        else:
            specified_task = "alltasks"
        
        train_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_{specified_task}.jsonl"
        # Dev data will be added to training
        dev_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl"
        # Test data for final predictions
        test_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_test_{task}.jsonl"

        try:
            # Load train data
            train_raw = load_jsonl_url(train_url)
            if train_raw:
                all_train.extend(train_raw)
                print(f"Loaded DimABSA Train: {lang}-{domain} ✅")

            # Load dev data and ADD TO TRAINING
            dev_raw = load_jsonl_url(dev_url)
            if dev_raw:
                all_train.extend(dev_raw)
                print(f"Loaded DimABSA Dev (for training): {lang}-{domain} ✅")
            
            # Load test data for predictions
            test_raw = load_jsonl_url(test_url)
            if test_raw:
                all_test.extend(test_raw)
                print(f"Loaded DimABSA Test: {lang}-{domain} ✅")
        except Exception as e:
            print(f"Skipped {lang}-{domain}: {e}")

# --- Summary ---
print(f"\n--- Data Loading Summary ---")
print(f"Total training samples (train + dev): {len(all_train)}")
print(f"Total test samples (for predictions): {len(all_test)}")

# --- 3. Load External SIGHAN 2024 Data (JSON) ---
print("\n--- Loading External SIGHAN 2024 Data ---")

sighan_urls = {
    "SIGHAN_Train1": "https://raw.githubusercontent.com/NYCU-NLP/SIGHAN2024-dimABSA/refs/heads/main/DataSets/dimABSA2024/Simplified/SIGHAN2024_dimABSA_TrainingSet1_Simplified.json",
    "SIGHAN_Train2": "https://raw.githubusercontent.com/NYCU-NLP/SIGHAN2024-dimABSA/refs/heads/main/DataSets/dimABSA2024/Simplified/SIGHAN2024_dimABSA_TrainingSet2_Simplified.json"
}

for name, url in sighan_urls.items():
    try:
        print(f"Fetching and processing {name}...")

        # 1. Fetch the raw data (it's a JSON file)
        external_raw_data = load_json_url(url) # Use NEW JSON loader

        if external_raw_data:
            # 2. Transform the raw data directly in memory
            external_transformed_data = transform_sighan_data(
                external_data=external_raw_data,
                # language="zho",
                # domain="restaurant" # SIGHAN data is restaurant domain
            )

            # 3. Add the new data to the main training list
            all_train.extend(external_transformed_data)
            print(f"Successfully added {len(external_transformed_data)} items from {name}.")

    except Exception as e:
        print(f"Failed to load or transform external data from {name}: {e}")

# # --- 2. Load augmented DimABSA 2026 Data (JSONL) ---
# print("--- Loading augmented DimABSA 2026 Data ---")
# for lang in ["eng_from_zho","zho_from_eng"]:
#     for domain in domains:
#         train_url = f"https://raw.githubusercontent.com/affan002/DeepLearning_project/refs/heads/main/augmented_datasets/augmented_translation_zho_to_eng/{lang}_{domain}.jsonl"

#         try:
#             train_raw = load_jsonl_url(train_url) # Use JSONL loader
#             if train_raw:
#               all_train.extend(train_raw)
#               print(f"Loaded DimABSA Train: {lang}-{domain} ✅")

#         except Exception as e:
#             print(f"Skipped {lang}-{domain}: {e}")



--- Loading DimABSA 2026 Data ---
Loaded DimABSA Train: eng-restaurant ✅
Loaded DimABSA Dev: eng-restaurant ✅
Loaded DimABSA Train: eng-laptop ✅
Loaded DimABSA Dev: eng-laptop ✅
Loaded DimABSA Train: zho-restaurant ✅
Loaded DimABSA Dev: zho-restaurant ✅
Loaded DimABSA Train: zho-laptop ✅
Loaded DimABSA Dev: zho-laptop ✅

--- Loading External SIGHAN 2024 Data ---
Fetching and processing SIGHAN_Train1...
Successfully added 3000 items from SIGHAN_Train1.
Fetching and processing SIGHAN_Train2...
Successfully added 3050 items from SIGHAN_Train2.
--- Loading augmented DimABSA 2026 Data ---
Loaded DimABSA Train: eng_from_zho-restaurant ✅
Loaded DimABSA Train: eng_from_zho-laptop ✅
Loaded DimABSA Train: zho_from_eng-restaurant ✅
Loaded DimABSA Train: zho_from_eng-laptop ✅


In [ ]:
# ============================================
# CELL 4.5: Clean all_train data
# ============================================
print("Cleaning all_train data for invalid Text or Aspect values...")

cleaned_all_train = []
removed_dp_count = 0
removed_quad_count = 0

for dp in all_train:
    # Check if the main 'Text' of the data point is valid
    if not isinstance(dp.get("Text"), str) or dp.get("Text") is None:
        removed_dp_count += 1
        continue # Skip this entire data point

    # Filter quadruplets within the data point
    filtered_quadruplets = []
    if "Quadruplet" in dp and isinstance(dp["Quadruplet"], list):
        for quad in dp["Quadruplet"]:
            if isinstance(quad.get("Aspect"), str) and quad.get("Aspect") is not None:
                filtered_quadruplets.append(quad)
            else:
                removed_quad_count += 1

    # Only add the data point if it still has valid quadruplets after filtering
    # or if it's a valid data point that might not have quadruplets (though our schema expects it).
    # For this specific error, we focus on quadruplets for Aspect.
    if filtered_quadruplets:
        new_dp = dp.copy()
        new_dp["Quadruplet"] = filtered_quadruplets
        cleaned_all_train.append(new_dp)
    else:
      # If a DP has no valid quadruplets after filtering, it's effectively useless for training.
      # Increment removed_dp_count if not already counted
      if not (not isinstance(dp.get("Text"), str) or dp.get("Text") is None):
          removed_dp_count += 1 # Count as removed if all quads were bad


all_train = cleaned_all_train

print(f"Finished cleaning. Removed {removed_dp_count} data points and {removed_quad_count} individual quadruplets.")
print(f"New size of all_train: {len(all_train)}")

Cleaning all_train data for invalid Text or Aspect values...
Finished cleaning. Removed 28 data points and 28 individual quadruplets.
New size of all_train: 37822


In [ ]:
# ============================================
# CELL 5: Prepare Entity Classification Data
# ============================================
entity_data = []
for dp in all_train:
    text = dp["Text"]
    for quad in dp["Quadruplet"]:
        aspect = quad["Aspect"]
        category = quad["Category"]
        # Split category into entity and attribute
        entity = category.split("#")[0]
        attribute = category.split("#")[1] if "#" in category else ""

        entity_data.append({
            "Text": text,
            "Aspect": aspect,
            "Entity": entity,
            "Attribute": attribute,
            "Category": category
        })

print(f"Total samples for Entity classification: {len(entity_data)}")
print(f"Sample: {entity_data[0]}")

Total samples for Entity classification: 48852
Sample: {'Text': "ca n ' t wait wait for my next visit .", 'Aspect': 'NULL', 'Entity': 'RESTAURANT', 'Attribute': 'GENERAL', 'Category': 'RESTAURANT#GENERAL'}


In [ ]:
# ============================================
# CELL 6: Create entity Dataset and Labels
# ============================================
# Get unique entities
unique_entities = sorted(list(set(d["Entity"] for d in entity_data)))
entity2id = {entity: i for i, entity in enumerate(unique_entities)}
id2entity = {i: entity for entity, i in entity2id.items()}

print(f"Unique Entities: {unique_entities}")
print(f"Entity to ID mapping: {entity2id}")

# Encode labels
for d in entity_data:
    d["EntityLabel"] = entity2id[d["Entity"]]

# Create dataset
entity_dataset = Dataset.from_list(entity_data)

Unique Entities: ['AMBIENCE', 'BATTERY', 'COMPANY', 'CPU', 'DISPLAY', 'DRINKS', 'FANS&COOLING', 'FANS_COOLING', 'FOOD', 'GRAPHICS', 'HARDWARE', 'HARD_DISC', 'HARD_DISK', 'KEYBOARD', 'LAPTOP', 'LOCATION', 'MEMORY', 'MOTHERBOARD', 'MOUSE', 'MULTIMEDIA_DEVICES', 'OPTICAL_DRIVES', 'OS', 'OUT_OF_SCOPE', 'Out_Of_Scope', 'PORTS', 'POWER_SUPPLY', 'RESTAURANT', 'SERVICE', 'SHIPPING', 'SOFTWARE', 'SUPPORT', 'WARRANTY', '地点', '服务', '氛围', '食物', '餐厅', '饮料']
Entity to ID mapping: {'AMBIENCE': 0, 'BATTERY': 1, 'COMPANY': 2, 'CPU': 3, 'DISPLAY': 4, 'DRINKS': 5, 'FANS&COOLING': 6, 'FANS_COOLING': 7, 'FOOD': 8, 'GRAPHICS': 9, 'HARDWARE': 10, 'HARD_DISC': 11, 'HARD_DISK': 12, 'KEYBOARD': 13, 'LAPTOP': 14, 'LOCATION': 15, 'MEMORY': 16, 'MOTHERBOARD': 17, 'MOUSE': 18, 'MULTIMEDIA_DEVICES': 19, 'OPTICAL_DRIVES': 20, 'OS': 21, 'OUT_OF_SCOPE': 22, 'Out_Of_Scope': 23, 'PORTS': 24, 'POWER_SUPPLY': 25, 'RESTAURANT': 26, 'SERVICE': 27, 'SHIPPING': 28, 'SOFTWARE': 29, 'SUPPORT': 30, 'WARRANTY': 31, '地点': 32, '服务':

In [ ]:
# ============================================
# CELL 7: Split Domain Dataset
# ============================================
# Split: 70% train, 20% validation, 10% test
split_datasets = entity_dataset.train_test_split(test_size=0.3, seed=42)
val_test_split = split_datasets["test"].train_test_split(test_size=1/3, seed=42)

entity_datasets = DatasetDict({
    "train": split_datasets["train"],
    "validation": val_test_split["train"],
    "test": val_test_split["test"]
})

print(f"Train: {len(entity_datasets['train'])}")
print(f"Validation: {len(entity_datasets['validation'])}")
print(f"Test: {len(entity_datasets['test'])}")

Train: 34196
Validation: 9770
Test: 4886


In [ ]:
# ============================================
# CELL 8: Tokenize Domain Dataset
# ============================================
checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_for_entity(example):
    return tokenizer(example["Text"], example["Aspect"], truncation=True, max_length=128)

tokenized_entity_datasets = entity_datasets.map(tokenize_for_entity, batched=True)
tokenized_entity_datasets = tokenized_entity_datasets.remove_columns(["Text", "Aspect", "Entity", "Attribute", "Category"])
tokenized_entity_datasets = tokenized_entity_datasets.rename_column("EntityLabel", "labels")
tokenized_entity_datasets.set_format("torch")

print(f"Tokenized columns: {tokenized_entity_datasets['train'].column_names}")


Map:   0%|          | 0/34196 [00:00<?, ? examples/s]

Map:   0%|          | 0/9770 [00:00<?, ? examples/s]

Map:   0%|          | 0/4886 [00:00<?, ? examples/s]

Tokenized columns: ['labels', 'input_ids', 'attention_mask']


In [ ]:
# ============================================
# CELL 9: Create entity DataLoaders
# ============================================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

entity_train_dataloader = DataLoader(
    tokenized_entity_datasets["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator
)
entity_eval_dataloader = DataLoader(
    tokenized_entity_datasets["validation"],
    batch_size=16,
    collate_fn=data_collator
)
entity_test_dataloader = DataLoader(
    tokenized_entity_datasets["test"],
    batch_size=16,
    collate_fn=data_collator
)

print(f"Training batches: {len(entity_train_dataloader)}")


Training batches: 2138


In [ ]:
# ============================================
# CELL 10: Initialize entity Model
# ============================================
num_entity_labels = len(entity2id)
print(f"Number of domain labels: {num_entity_labels}")

entity_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_entity_labels
)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
entity_model.to(device)
print(f"Using device: {device}")

Number of domain labels: 38


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [ ]:
# ============================================
# CELL 11: Setup entity Training
# ============================================
from torch.optim import AdamW

optimizer = AdamW(entity_model.parameters(), lr=2e-5)

num_epochs = 5
num_training_steps = num_epochs * len(entity_train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

print(f"Total training steps: {num_training_steps}")

Total training steps: 2138


In [ ]:
# ============================================
# CELL 12: Train entity Model
# ============================================
progress_bar = tqdm(range(num_training_steps))

entity_model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in entity_train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = entity_model(**batch)
        loss = outputs.loss
        epoch_loss += loss.item()

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    avg_loss = epoch_loss / len(entity_train_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")


  0%|          | 0/2138 [00:00<?, ?it/s]

Epoch 1/1, Loss: 0.8571


In [ ]:
# ============================================
# CELL 13: Evaluate entity Model
# ============================================
metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

entity_model.eval()
for batch in entity_eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = entity_model(**batch)
    predictions = torch.argmax(outputs.logits, dim=-1)

    metric_f1.add_batch(predictions=predictions, references=batch["labels"])
    metric_acc.add_batch(predictions=predictions, references=batch["labels"])

f1_score = metric_f1.compute(average="weighted")
accuracy = metric_acc.compute()

print(f"\nentity Model Results:")
print(f"Accuracy: {accuracy['accuracy']:.4f}")
print(f"F1 Score: {f1_score['f1']:.4f}")


entity Model Results:
Accuracy: 0.8810
F1 Score: 0.8669


In [ ]:
# ============================================
# CELL 14: Save entity Model
# ============================================
entity_model.save_pretrained("./entity_classifier")
tokenizer.save_pretrained("./entity_classifier")
print("entity model saved!")

entity model saved!


In [ ]:
# ============================================
# CELL 15: Prepare Attribute Classification Data
# ============================================
# Now prepare data for attribute classification
# We'll use: Text + Aspect + Predicted entity -> Attribute

attribute_data = []
for dp in all_train:
    text = dp["Text"]
    for quad in dp["Quadruplet"]:
        aspect = quad["Aspect"]
        category = quad["Category"]
        entity = category.split("#")[0]
        attribute = category.split("#")[1] if "#" in category else ""

        if attribute:  # Only include samples with attributes
            attribute_data.append({
                "Text": text,
                "Aspect": aspect,
                "Entity": entity,
                "Attribute": attribute
            })

print(f"Total samples for attribute classification: {len(attribute_data)}")
print(f"Sample: {attribute_data[0]}")

Total samples for attribute classification: 48852
Sample: {'Text': "ca n ' t wait wait for my next visit .", 'Aspect': 'NULL', 'Entity': 'RESTAURANT', 'Attribute': 'GENERAL'}


In [ ]:
# ============================================
# CELL 16: Create Attribute Dataset and Labels
# ============================================
# Get unique attributes
unique_attributes = sorted(list(set(d["Attribute"] for d in attribute_data)))
attribute2id = {attr: i for i, attr in enumerate(unique_attributes)}
id2attribute = {i: attr for attr, i in attribute2id.items()}

print(f"Unique Attributes: {unique_attributes}")
print(f"Number of attributes: {len(unique_attributes)}")

# Encode labels
for d in attribute_data:
    d["AttributeLabel"] = attribute2id[d["Attribute"]]
    d["EntityLabel"] = entity2id[d["Entity"]]

# Create dataset
attribute_dataset = Dataset.from_list(attribute_data)


Unique Attributes: ['CONNECTIVITY', 'DESIGN_FEATURES', 'GENERAL', 'MISCELLANEOUS', 'OPERATION_PERFORMANCE', 'PORTABILITY', 'PRICE', 'PRICES', 'QUALITY', 'STYLE_OPTIONS', 'USABILITY', '价格', '份量与款式', '品质', '杂项', '概括']
Number of attributes: 16


In [ ]:
# ============================================
# CELL 17: Split Attribute Dataset
# ============================================
split_attr_datasets = attribute_dataset.train_test_split(test_size=0.3, seed=42)
val_test_attr_split = split_attr_datasets["test"].train_test_split(test_size=1/3, seed=42)

attribute_datasets = DatasetDict({
    "train": split_attr_datasets["train"],
    "validation": val_test_attr_split["train"],
    "test": val_test_attr_split["test"]
})

print(f"Attribute Train: {len(attribute_datasets['train'])}")
print(f"Attribute Validation: {len(attribute_datasets['validation'])}")
print(f"Attribute Test: {len(attribute_datasets['test'])}")


Attribute Train: 34196
Attribute Validation: 9770
Attribute Test: 4886


In [ ]:
# ============================================
# CELL 18: Tokenize Attribute Dataset
# ============================================
def tokenize_for_attribute(examples):
    # Combine Text, Aspect, and Entity as input for each example in the batch
    combined_texts = [
        f"{text} [SEP] {aspect} [SEP] {entity}"
        for text, aspect, entity in zip(examples['Text'], examples['Aspect'], examples['Entity'])
    ]
    return tokenizer(combined_texts, truncation=True, max_length=128, padding=False)

tokenized_attr_datasets = attribute_datasets.map(tokenize_for_attribute, batched=True)
tokenized_attr_datasets = tokenized_attr_datasets.remove_columns(["Text", "Aspect", "Entity", "Attribute", "EntityLabel"])
tokenized_attr_datasets = tokenized_attr_datasets.rename_column("AttributeLabel", "labels")
tokenized_attr_datasets.set_format("torch")

Map:   0%|          | 0/34196 [00:00<?, ? examples/s]

Map:   0%|          | 0/9770 [00:00<?, ? examples/s]

Map:   0%|          | 0/4886 [00:00<?, ? examples/s]

In [ ]:
# ============================================
# CELL 19: Create Attribute DataLoaders
# ============================================
attr_train_dataloader = DataLoader(
    tokenized_attr_datasets["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator
)
attr_eval_dataloader = DataLoader(
    tokenized_attr_datasets["validation"],
    batch_size=16,
    collate_fn=data_collator
)
attr_test_dataloader = DataLoader(
    tokenized_attr_datasets["test"],
    batch_size=16,
    collate_fn=data_collator
)

In [ ]:
# ============================================
# CELL 20: Initialize Attribute Model
# ============================================
num_attribute_labels = len(attribute2id)
print(f"Number of attribute labels: {num_attribute_labels}")

attribute_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_attribute_labels
)
attribute_model.to(device)

Number of attribute labels: 16


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [ ]:
# ============================================
# CELL 21: Setup Attribute Training
# ============================================
attr_optimizer = AdamW(attribute_model.parameters(), lr=2e-5)

attr_num_epochs = 5
attr_num_training_steps = attr_num_epochs * len(attr_train_dataloader)
attr_lr_scheduler = get_scheduler(
    "linear",
    optimizer=attr_optimizer,
    num_warmup_steps=int(0.1 * attr_num_training_steps),
    num_training_steps=attr_num_training_steps,
)

print(f"Attribute training steps: {attr_num_training_steps}")


Attribute training steps: 2138


In [ ]:
# ============================================
# CELL 22: Train Attribute Model
# ============================================
attr_progress_bar = tqdm(range(attr_num_training_steps))

attribute_model.train()
for epoch in range(attr_num_epochs):
    epoch_loss = 0
    for batch in attr_train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = attribute_model(**batch)
        loss = outputs.loss
        epoch_loss += loss.item()

        loss.backward()
        attr_optimizer.step()
        attr_lr_scheduler.step()
        attr_optimizer.zero_grad()
        attr_progress_bar.update(1)

    avg_loss = epoch_loss / len(attr_train_dataloader)
    print(f"Epoch {epoch+1}/{attr_num_epochs}, Loss: {avg_loss:.4f}")


  0%|          | 0/2138 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================
# CELL 23: Evaluate Attribute Model
# ============================================
metric_f1_attr = evaluate.load("f1")
metric_acc_attr = evaluate.load("accuracy")

attribute_model.eval()
for batch in attr_eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = attribute_model(**batch)
    predictions = torch.argmax(outputs.logits, dim=-1)

    metric_f1_attr.add_batch(predictions=predictions, references=batch["labels"])
    metric_acc_attr.add_batch(predictions=predictions, references=batch["labels"])

f1_score_attr = metric_f1_attr.compute(average="weighted")
accuracy_attr = metric_acc_attr.compute()

print(f"\nAttribute Model Results:")
print(f"Accuracy: {accuracy_attr['accuracy']:.4f}")
print(f"F1 Score: {f1_score_attr['f1']:.4f}")


In [ ]:
# ============================================
# CELL 24: Save Attribute Model
# ============================================
attribute_model.save_pretrained("./attribute_classifier")
print("Attribute model saved!")

# Save label mappings
import pickle

with open('label_mappings.pkl', 'wb') as f:
    pickle.dump({
        'entity2id': entity2id,
        'id2entity': id2entity,
        'attribute2id': attribute2id,
        'id2attribute': id2attribute
    }, f)
print("Label mappings saved!")

In [ ]:
# ============================================
# CELL 25: Inference Pipeline
# ============================================
def predict_category(text, aspect):
    """
    Predict the full category (Entity#Attribute) for a given text and aspect.
    """
    # Step 1: Predict Entity
    entity_model.eval()
    inputs = tokenizer(text, aspect, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = entity_model(**inputs)
    entity_id = torch.argmax(outputs.logits, dim=-1).item()
    entity = id2entity[entity_id]

    # Step 2: Predict Attribute
    attribute_model.eval()
    combined_text = f"{text} [SEP] {aspect} [SEP] {entity}"
    inputs = tokenizer(combined_text, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = attribute_model(**inputs)
    attribute_id = torch.argmax(outputs.logits, dim=-1).item()
    attribute = id2attribute[attribute_id]

    category = f"{entity}#{attribute}"
    return entity, attribute, category

# Test the pipeline
test_text = "this unit is pretty and stylish, so my high school daughter was attracted to it for that reason."
test_aspect = "unit"

entity, attribute, category = predict_category(test_text, test_aspect)
print(f"\nTest Prediction:")
print(f"Text: {test_text}")
print(f"Aspect: {test_aspect}")
print(f"Predicted Entity: {entity}")
print(f"Predicted Attribute: {attribute}")
print(f"Predicted Category: {category}")

In [ ]:
# ============================================
# CELL 27: Final Evaluation on Held-Out Test Data
# ============================================
print("\n" + "="*60)
print("FINAL EVALUATION ON HELD-OUT TEST DATA (10%)")
print("="*60)

def evaluate_on_held_out_test():
    """
    Evaluate the complete pipeline on the 10% held-out test data
    that was never seen during training or validation.
    """

    # Get the original data back from test splits
    entity_test_data = tokenized_entity_datasets["test"]
    attr_test_data = tokenized_attr_datasets["test"]

    print(f"\nEntity Test Set Size: {len(entity_test_data)}")
    print(f"Attribute Test Set Size: {len(attr_test_data)}")

    # ===== Step 1: Evaluate entity Model on Test Set =====
    print("\n--- Step 1: Evaluating entity Model ---")

    entity_model.eval()
    metric_f1_entity = evaluate.load("f1")
    metric_acc_entity= evaluate.load("accuracy")

    all_entity_preds = []
    all_entity_labels = []

    for batch in tqdm(entity_test_dataloader, desc="entity predictions"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = entity_model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)

        all_entity_preds.extend(predictions.cpu().numpy())
        all_entity_labels.extend(batch["labels"].cpu().numpy())

        metric_f1_entity.add_batch(predictions=predictions, references=batch["labels"])
        metric_acc_entity.add_batch(predictions=predictions, references=batch["labels"])

    entity_f1 = metric_f1_entity.compute(average="weighted")
    entity_acc = metric_acc_entity.compute()

    print(f"entity Model - Accuracy: {entity_acc['accuracy']:.4f}")
    print(f"entity Model - F1 Score: {entity_f1['f1']:.4f}")

    # ===== Step 2: Evaluate Attribute Model on Test Set =====
    print("\n--- Step 2: Evaluating Attribute Model ---")

    attribute_model.eval()
    metric_f1_attr = evaluate.load("f1")
    metric_acc_attr = evaluate.load("accuracy")

    all_attr_preds = []
    all_attr_labels = []

    for batch in tqdm(attr_test_dataloader, desc="Attribute predictions"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = attribute_model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)

        all_attr_preds.extend(predictions.cpu().numpy())
        all_attr_labels.extend(batch["labels"].cpu().numpy())

        metric_f1_attr.add_batch(predictions=predictions, references=batch["labels"])
        metric_acc_attr.add_batch(predictions=predictions, references=batch["labels"])

    attr_f1 = metric_f1_attr.compute(average="weighted")
    attr_acc = metric_acc_attr.compute()

    print(f"Attribute Model - Accuracy: {attr_acc['accuracy']:.4f}")
    print(f"Attribute Model - F1 Score: {attr_f1['f1']:.4f}")

    # ===== Step 3: End-to-End Pipeline Evaluation =====
    print("\n--- Step 3: End-to-End Pipeline Evaluation ---")
    print("(Predicting Entity first, then using it to predict Attribute)")

    # We need to reconstruct samples from the original data
    # Get samples that are in the test split
    test_samples_for_pipeline = []

    # Use the attribute test data since it has both entity and attribute info
    # We'll get the original text/aspect from domain_data
    test_indices = set(range(int(0.9 * len(entity_data)), len(entity_data)))

    # Collect test samples from original domain_data
    for idx in list(test_indices)[:200]:  # Limit to 200 samples for speed
        if idx < len(entity_data):
            test_samples_for_pipeline.append(entity_data[idx])

    print(f"Testing pipeline on {len(test_samples_for_pipeline)} samples...")

    pipeline_results = {
        'entity_correct': 0,
        'attribute_correct': 0,
        'both_correct': 0,
        'total': 0
    }

    detailed_results = []

    for sample in tqdm(test_samples_for_pipeline, desc="Pipeline evaluation"):
        text = sample['Text']
        aspect = sample['Aspect']
        true_entity = sample['Entity']
        true_attribute = sample['Attribute']

        # Predict using the pipeline
        pred_entity, pred_attribute, pred_category = predict_category(text, aspect)

        entity_match = (pred_entity == true_entity)
        attr_match = (pred_attribute == true_attribute)
        both_match = entity_match and attr_match

        pipeline_results['entity_correct'] += int(entity_match)
        pipeline_results['attribute_correct'] += int(attr_match)
        pipeline_results['both_correct'] += int(both_match)
        pipeline_results['total'] += 1

        # Store some examples for inspection
        if len(detailed_results) < 10:
            detailed_results.append({
                'text': text[:80] + '...' if len(text) > 80 else text,
                'aspect': aspect,
                'true': f"{true_entity}#{true_attribute}",
                'pred': pred_category,
                'correct': '✓' if both_match else '✗'
            })

    # Calculate final metrics
    entity_acc_pipeline = pipeline_results['entity_correct'] / pipeline_results['total']
    attr_acc_pipeline = pipeline_results['attribute_correct'] / pipeline_results['total']
    full_acc_pipeline = pipeline_results['both_correct'] / pipeline_results['total']

    print(f"\n{'='*60}")
    print("PIPELINE RESULTS (Entity → Attribute):")
    print(f"{'='*60}")
    print(f"Entity Prediction Accuracy:    {entity_acc_pipeline:.4f} ({pipeline_results['entity_correct']}/{pipeline_results['total']})")
    print(f"Attribute Prediction Accuracy: {attr_acc_pipeline:.4f} ({pipeline_results['attribute_correct']}/{pipeline_results['total']})")
    print(f"Full Category Accuracy:        {full_acc_pipeline:.4f} ({pipeline_results['both_correct']}/{pipeline_results['total']})")
    print(f"{'='*60}")

    # Show some example predictions
    print("\n--- Sample Predictions ---")
    for i, result in enumerate(detailed_results, 1):
        print(f"\n{i}. {result['correct']}")
        print(f"   Text: {result['text']}")
        print(f"   Aspect: {result['aspect']}")
        print(f"   True: {result['true']}")
        print(f"   Pred: {result['pred']}")

    # ===== Summary =====
    print(f"\n{'='*60}")
    print("SUMMARY - TEST SET PERFORMANCE")
    print(f"{'='*60}")
    print(f"Individual Model Performance:")
    print(f"  entity Model:    Acc={entity_acc['accuracy']:.4f}, F1={entity_f1['f1']:.4f}")
    print(f"  Attribute Model: Acc={attr_acc['accuracy']:.4f}, F1={attr_f1['f1']:.4f}")
    print(f"\nPipeline Performance (Entity→Attribute):")
    print(f"  Entity Stage:    {entity_acc_pipeline:.4f}")
    print(f"  Attribute Stage: {attr_acc_pipeline:.4f}")
    print(f"  Full Pipeline:   {full_acc_pipeline:.4f}")
    print(f"{'='*60}")

    return {
        'entity_metrics': {'accuracy': entity_acc['accuracy'], 'f1': entity_f1['f1']},
        'attribute_metrics': {'accuracy': attr_acc['accuracy'], 'f1': attr_f1['f1']},
        'pipeline_metrics': {
            'entity_accuracy': entity_acc_pipeline,
            'attribute_accuracy': attr_acc_pipeline,
            'full_accuracy': full_acc_pipeline
        },
        'detailed_results': detailed_results
    }

# Run the comprehensive evaluation
test_results = evaluate_on_held_out_test()

In [ ]:
from huggingface_hub import HfApi
import os
from huggingface_hub import login
import json

login(token=os.environ["HF_TOKEN"])

entity_repo_id = f"hassanshahzad2003/{checkpoint}_task3_entity_aug"
attr_repo_id = f"hassanshahzad2003/{checkpoint}_task3_attribute_aug"

# ===== Upload Entity Model =====
entity_save_dir = "hf_entity_model"
os.makedirs(entity_save_dir, exist_ok=True)

# Save the FULL model (config + weights)
entity_model.save_pretrained(entity_save_dir)
print(f"✅ Entity model saved to {entity_save_dir}")

# Save tokenizer
tokenizer.save_pretrained(entity_save_dir)
print(f"✅ Tokenizer saved to {entity_save_dir}")

# Save label mappings
with open(os.path.join(entity_save_dir, "label_mappings.json"), "w") as f:
    json.dump({"entity2id": entity2id, "id2entity": id2entity}, f, indent=2)
print(f"✅ Label mappings saved")

# Upload to Hub
api = HfApi()
api.create_repo(repo_id=entity_repo_id, private=True, exist_ok=True)
api.upload_folder(
    folder_path=entity_save_dir,
    repo_id=entity_repo_id,
    repo_type="model",
    commit_message="Upload entity classifier model with config"
)
print(f"✅ Entity model uploaded: https://huggingface.co/{entity_repo_id}")

# ===== Upload Attribute Model =====
attr_save_dir = "hf_attribute_model"
os.makedirs(attr_save_dir, exist_ok=True)

# Save the FULL model (config + weights)
attribute_model.save_pretrained(attr_save_dir)
print(f"✅ Attribute model saved to {attr_save_dir}")

# Save tokenizer
tokenizer.save_pretrained(attr_save_dir)
print(f"✅ Tokenizer saved to {attr_save_dir}")

# Save label mappings
with open(os.path.join(attr_save_dir, "label_mappings.json"), "w") as f:
    json.dump({"attribute2id": attribute2id, "id2attribute": id2attribute}, f, indent=2)
print(f"✅ Label mappings saved")

# Upload to Hub
api.create_repo(repo_id=attr_repo_id, private=True, exist_ok=True)
api.upload_folder(
    folder_path=attr_save_dir,
    repo_id=attr_repo_id,
    repo_type="model",
    commit_message="Upload attribute classifier model with config"
)
print(f"✅ Attribute model uploaded: https://huggingface.co/{attr_repo_id}")

print("\n" + "="*60)
print("🎉 BOTH MODELS UPLOADED SUCCESSFULLY!")
print("="*60)
print(f"Entity Model:    https://huggingface.co/{entity_repo_id}")
print(f"Attribute Model: https://huggingface.co/{attr_repo_id}")

✅ Entity model saved to hf_entity_model
✅ Tokenizer saved to hf_entity_model
✅ Label mappings saved


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Entity model uploaded: https://huggingface.co/hassanshahzad2003/xlm-roberta-base_task3_entity
✅ Attribute model saved to hf_attribute_model
✅ Tokenizer saved to hf_attribute_model
✅ Label mappings saved


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Attribute model uploaded: https://huggingface.co/hassanshahzad2003/xlm-roberta-base_task3_attribute

🎉 BOTH MODELS UPLOADED SUCCESSFULLY!
Entity Model:    https://huggingface.co/hassanshahzad2003/xlm-roberta-base_task3_entity
Attribute Model: https://huggingface.co/hassanshahzad2003/xlm-roberta-base_task3_attribute


In [ ]:
# ============================================
# CELL: Load Models from Hugging Face and Evaluate
# ============================================
from huggingface_hub import hf_hub_download , login
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json
import torch

# Login with your token for private models
login(token=os.environ["HF_TOKEN"])  # <--- Add your token here
print("✅ Logged in to Hugging Face")

print("="*60)
print("LOADING MODELS FROM HUGGING FACE HUB")
print("="*60)

# Define your model repository IDs
checkpoint = "xlm-roberta-base"  # or whatever checkpoint you used
entity_repo_id = f"hassanshahzad2003/{checkpoint}_task3_entity_aug"
attr_repo_id = f"hassanshahzad2003/{checkpoint}_task3_attribute_aug"

# ===== Load Entity Model =====
print("\n--- Loading Entity Model ---")
entity_model_loaded = AutoModelForSequenceClassification.from_pretrained(entity_repo_id)
tokenizer_loaded = AutoTokenizer.from_pretrained(entity_repo_id)

# Download and load label mappings
entity_label_file = hf_hub_download(repo_id=entity_repo_id, filename="label_mappings.json")
with open(entity_label_file, 'r') as f:
    entity_labels = json.load(f)
    entity2id_loaded = entity_labels['entity2id']
    id2entity_loaded = {int(k): v for k, v in entity_labels['id2entity'].items()}

print(f"✅ Entity model loaded from: {entity_repo_id}")
print(f"   Entities: {list(entity2id_loaded.keys())}")

# ===== Load Attribute Model =====
print("\n--- Loading Attribute Model ---")
attribute_model_loaded = AutoModelForSequenceClassification.from_pretrained(attr_repo_id)

# Download and load label mappings
attr_label_file = hf_hub_download(repo_id=attr_repo_id, filename="label_mappings.json")
with open(attr_label_file, 'r') as f:
    attr_labels = json.load(f)
    attribute2id_loaded = attr_labels['attribute2id']
    id2attribute_loaded = {int(k): v for k, v in attr_labels['id2attribute'].items()}

print(f"✅ Attribute model loaded from: {attr_repo_id}")
print(f"   Attributes: {len(attribute2id_loaded)} classes")

# Move models to device
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
entity_model_loaded.to(device)
attribute_model_loaded.to(device)
print(f"\n✅ Models moved to: {device}")

# ===== Update Prediction Function to Use Loaded Models =====
def predict_category_loaded(text, aspect):
    """
    Predict the full category (Entity#Attribute) using loaded models from HF Hub.
    """
    # Step 1: Predict Entity
    entity_model_loaded.eval()
    inputs = tokenizer_loaded(text, aspect, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = entity_model_loaded(**inputs)
    entity_id = torch.argmax(outputs.logits, dim=-1).item()
    entity = id2entity_loaded[entity_id]

    # Step 2: Predict Attribute
    attribute_model_loaded.eval()
    combined_text = f"{text} [SEP] {aspect} [SEP] {entity}"
    inputs = tokenizer_loaded(combined_text, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = attribute_model_loaded(**inputs)
    attribute_id = torch.argmax(outputs.logits, dim=-1).item()
    attribute = id2attribute_loaded[attribute_id]

    category = f"{entity}#{attribute}"
    return entity, attribute, category

# ===== Test Single Prediction =====
print("\n--- Testing Loaded Models ---")
test_text = "this unit is pretty and stylish, so my high school daughter was attracted to it for that reason."
test_aspect = "unit"

entity, attribute, category = predict_category_loaded(test_text, test_aspect)
print(f"\nTest Prediction:")
print(f"Text: {test_text}")
print(f"Aspect: {test_aspect}")
print(f"Predicted Entity: {entity}")
print(f"Predicted Attribute: {attribute}")
print(f"Predicted Category: {category}")

# ===== Run Full Evaluation with Loaded Models =====
print("\n" + "="*60)
print("RUNNING EVALUATION WITH LOADED MODELS")
print("="*60)

# Update global variables to use loaded models for evaluation
entity_model = entity_model_loaded
attribute_model = attribute_model_loaded
tokenizer = tokenizer_loaded
entity2id = entity2id_loaded
id2entity = id2entity_loaded
attribute2id = attribute2id_loaded
id2attribute = id2attribute_loaded

# Update predict_category function to use loaded models
predict_category = predict_category_loaded

# Run the evaluation
test_results_loaded = evaluate_on_held_out_test()

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE WITH LOADED MODELS!")
print("="*60)

✅ Logged in to Hugging Face
LOADING MODELS FROM HUGGING FACE HUB

--- Loading Entity Model ---


config.json:   0%|          | 0.00/1.95k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

entity Model - Accuracy: 0.9191
entity Model - F1 Score: 0.9187

--- Step 2: Evaluating Attribute Model ---


Attribute predictions:   0%|          | 0/153 [00:00<?, ?it/s]

Attribute Model - Accuracy: 0.8262
Attribute Model - F1 Score: 0.8254

--- Step 3: End-to-End Pipeline Evaluation ---
(Predicting Entity first, then using it to predict Attribute)
Testing pipeline on 200 samples...


Pipeline evaluation:   0%|          | 0/200 [00:00<?, ?it/s]


PIPELINE RESULTS (Entity → Attribute):
Entity Prediction Accuracy:    0.9250 (185/200)
Attribute Prediction Accuracy: 0.8800 (176/200)
Full Category Accuracy:        0.8200 (164/200)

--- Sample Predictions ---

1. ✓
   Text: 也買這台給老婆用，跟大廠比沒有比較差，效能好，筆電又輕又持久，不想花大錢這台真的首選~
   Aspect: 效能
   True: LAPTOP#OPERATION_PERFORMANCE
   Pred: LAPTOP#OPERATION_PERFORMANCE

2. ✓
   Text: 也買這台給老婆用，跟大廠比沒有比較差，效能好，筆電又輕又持久，不想花大錢這台真的首選~
   Aspect: 筆電
   True: LAPTOP#PORTABILITY
   Pred: LAPTOP#PORTABILITY

3. ✗
   Text: 也買這台給老婆用，跟大廠比沒有比較差，效能好，筆電又輕又持久，不想花大錢這台真的首選~
   Aspect: 筆電
   True: LAPTOP#OPERATION_PERFORMANCE
   Pred: LAPTOP#PORTABILITY

4. ✓
   Text: 有獨立的效能模式切換鍵變換時還蠻方便的
   Aspect: 有獨立的效能模式切換鍵變換
   True: KEYBOARD#DESIGN_FEATURES
   Pred: KEYBOARD#DESIGN_FEATURES

5. ✓
   Text: 等!不要急著買庫存~i13CPU和4系列顯卡會好很多!!!
   Aspect: i13CPU
   True: CPU#GENERAL
   Pred: CPU#GENERAL

6. ✓
   Text: 等!不要急著買庫存~i13CPU和4系列顯卡會好很多!!!
   Aspect: 4系列顯卡
   True: GRAPHICS#GENERAL
   Pred: GRAPHICS#GENERAL

7. ✓
   Text: 電競筆電就夠重了，

In [ ]:
# # ============================================
# # CELL 26: Batch Evaluation on Test Set
# # ============================================
# def evaluate_full_pipeline(test_samples, num_samples=50):
#     """Evaluate the full pipeline on test samples."""
#     results = []

#     for i, sample in enumerate(test_samples[:num_samples]):
#         text = sample["Text"]

#         # Handle both "Quadruplet" and "quadruplets" keys
#         quads = sample.get("Quadruplet") or sample.get("quadruplets") or sample.get("Quadruplets") or []

#         for quad in quads:
#             aspect = quad.get("Aspect") or quad.get("aspect", "")
#             true_category = quad.get("Category") or quad.get("category", "")

#             if not true_category or "#" not in true_category:
#                 continue  # Skip if no valid category

#             true_entity = true_category.split("#")[0]
#             true_attribute = true_category.split("#")[1] if "#" in true_category else ""

#             pred_entity, pred_attribute, pred_category = predict_category(text, aspect)

#             results.append({
#                 "text": text,
#                 "aspect": aspect,
#                 "true_category": true_category,
#                 "pred_category": pred_category,
#                 "entity_correct": pred_entity == true_entity,
#                 "attribute_correct": pred_attribute == true_attribute,
#                 "full_correct": pred_category == true_category
#             })

#     if len(results) == 0:
#         print("No valid results found. Check data format.")
#         return []

#     # Calculate metrics
#     entity_acc = sum(r["entity_correct"] for r in results) / len(results)
#     attribute_acc = sum(r["attribute_correct"] for r in results) / len(results)
#     full_acc = sum(r["full_correct"] for r in results) / len(results)

#     print(f"\nFull Pipeline Evaluation:")
#     print(f"Samples evaluated: {len(results)}")
#     print(f"Entity Accuracy: {entity_acc:.4f}")
#     print(f"Attribute Accuracy: {attribute_acc:.4f}")
#     print(f"Full Category Accuracy: {full_acc:.4f}")

#     return results

# # Run evaluation
# if all_dev:
#     print(f"Sample data structure: {list(all_dev[0].keys())}")
#     eval_results = evaluate_full_pipeline(all_dev, num_samples=100)